# Taller de Optimización Lineal: Despacho Eléctrico y Portafolio de Crédito

Este notebook contiene la formulación matemática y la implementación en Python (usando SciPy y PuLP) para resolver y analizar dos problemas de programación lineal, incluyendo su análisis de dualidad y sensibilidad detallado.


## PROBLEMA 1: Despacho Eléctrico

### 1. Formulación Primal (Matemática)
Sea $x_j$ la generación de la planta $j$ en GWh/semana para $j \in \{1, 2, 3, 4, 5\}$ (Hidro, Eólica, Gas, Carbón, Diésel).

**Función Objetivo (Minimizar Costo Variable Total en miles USD/semana):**
$$\min z = 12x_1 + 8x_2 + 45x_3 + 32x_4 + 95x_5$$

**Restricciones:**
1. **Demanda:**
   $$x_1 + x_2 + x_3 + x_4 + x_5 = 780 \quad (\text{GWh}) \quad [y_1 \in \mathbb{R}]$$
2. **Emisiones:**
   $$0x_1 + 0x_2 + 400x_3 + 900x_4 + 700x_5 \le 245000 \quad (\text{tCO}_2) \quad [y_2 \le 0]$$
3. **Cuota Renovable:**
   $$x_1 + x_2 \ge 312 \iff -x_1 - x_2 \le -312 \quad (\text{GWh}) \quad [y_3 \ge 0]$$
4. **Take-or-pay Gas:**
   $$x_3 \ge 50 \iff -x_3 \le -50 \quad (\text{GWh}) \quad [y_4 \ge 0]$$
5. **Límites de Capacidad:**
   $$0 \le x_1 \le 260 \quad [\mu_1^- \ge 0, \mu_1^+ \ge 0]$$
   $$0 \le x_2 \le 120 \quad [\mu_2^- \ge 0, \mu_2^+ \ge 0]$$
   $$0 \le x_3 \le 300 \quad [\mu_3^- \ge 0, \mu_3^+ \ge 0]$$
   $$0 \le x_4 \le 280 \quad [\mu_4^- \ge 0, \mu_4^+ \ge 0]$$
   $$0 \le x_5 \le 90 \quad [\mu_5^- \ge 0, \mu_5^+ \ge 0]$$

### 2. Formulación Dual
Sean las variables duales asociadas:
* $y_1$ para la Demanda (igualdad $\to$ no restringida).
* $y_2$ para Emisiones (restricción $\le$ en minimización $\to y_2 \le 0$).
* $y_3$ para Cuota Renovable (restricción $\ge$ en minimización $\to y_3 \ge 0$).
* $y_4$ para Take-or-pay (restricción $\ge$ en minimización $\to y_4 \ge 0$).
* $w_j$ para límites de capacidad superior (restricciones $x_j \le \text{cap}_j \to w_j \le 0$).

**Función Objetivo Dual (Maximizar):**
$$\max g = 780y_1 + 245000y_2 + 312y_3 + 50y_4 + \sum_{j=1}^5 \text{cap}_j w_j$$

**Restricciones Duales (asociadas a variables primales $x_j \ge 0$):**
1. **Para $x_1$:** $y_1 + 0y_2 + y_3 + 0y_4 + w_1 \le 12$
2. **Para $x_2$:** $y_1 + 0y_2 + y_3 + 0y_4 + w_2 \le 8$
3. **Para $x_3$:** $y_1 + 400y_2 + 0y_3 + y_4 + w_3 \le 45$
4. **Para $x_4$:** $y_1 + 900y_2 + 0y_3 + 0y_4 + w_4 \le 32$
5. **Para $x_5$:** $y_1 + 700y_2 + 0y_3 + 0y_4 + w_5 \le 95$

Con $y_1 \in \mathbb{R}$, $y_2 \le 0$, $y_3 \ge 0$, $y_4 \ge 0$, $w_j \le 0$.
```

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog
!pip install pulp -q
import pulp

# --- DATOS PROBLEMA 1 ---
c_p1 = np.array([12, 8, 45, 32, 95])

# Restricciones de desigualdad (A_ub * x <= b_ub)
# 1) Emisiones: 0*x1 + 0*x2 + 400*x3 + 900*x4 + 700*x5 <= 245000
# 2) Cuota renovable: -x1 - x2 <= -312
# 3) Take-or-pay gas: -x3 <= -50
A_ub_p1 = np.array([
    [0, 0, 400, 900, 700],
    [-1, -1, 0, 0, 0],
    [0, 0, -1, 0, 0]
])
b_ub_p1 = np.array([245000, -312, -50])

# Restricciones de igualdad (A_eq * x == b_eq)
# Demanda: x1 + x2 + x3 + x4 + x5 = 780
A_eq_p1 = np.array([[1, 1, 1, 1, 1]])
b_eq_p1 = np.array([780])

# Límites de capacidad
bounds_p1 = [
    (0, 260),  # G1
    (0, 120),  # G2
    (0, 300),  # G3
    (0, 280),  # G4
    (0, 90)    # G5
]

In [2]:
# --- RESOLUCIÓN PRIMAL CON SCIPY ---
res_p1_scipy = linprog(
    c=c_p1,
    A_ub=A_ub_p1,
    b_ub=b_ub_p1,
    A_eq=A_eq_p1,
    b_eq=b_eq_p1,
    bounds=bounds_p1,
    method='highs'
)

print("=== SOLUCIÓN PRIMAL SCIPY (PROB 1) ===")
print(f"Costo óptimo: {res_p1_scipy.fun:.4f} miles USD/semana")
print(f"Generación (G1-G5) GWh: {res_p1_scipy.x}")

=== SOLUCIÓN PRIMAL SCIPY (PROB 1) ===
Costo óptimo: 19870.0000 miles USD/semana
Generación (G1-G5) GWh: [260. 120. 230. 170.   0.]


In [3]:
# --- RESOLUCIÓN PRIMAL CON PULP ---
prob1 = pulp.LpProblem("Despacho_Electrico", pulp.LpMinimize)

x1 = pulp.LpVariable('x_G1', lowBound=0, upBound=260)
x2 = pulp.LpVariable('x_G2', lowBound=0, upBound=120)
x3 = pulp.LpVariable('x_G3', lowBound=0, upBound=300)
x4 = pulp.LpVariable('x_G4', lowBound=0, upBound=280)
x5 = pulp.LpVariable('x_G5', lowBound=0, upBound=90)

vars_p1 = [x1, x2, x3, x4, x5]

# Objetivo
prob1 += 12*x1 + 8*x2 + 45*x3 + 32*x4 + 95*x5

# Restricciones
prob1 += (x1 + x2 + x3 + x4 + x5 == 780, "Demanda")
prob1 += (400*x3 + 900*x4 + 700*x5 <= 245000, "Emisiones")
prob1 += (x1 + x2 >= 312, "Cuota_Renovable")
prob1 += (x3 >= 50, "Take_or_pay")

prob1.solve(pulp.PULP_CBC_CMD(msg=False))

print("=== SOLUCIÓN PRIMAL PULP (PROB 1) ===")
print(f"Estado: {pulp.LpStatus[prob1.status]}")
print(f"Costo óptimo: {pulp.value(prob1.objective):.4f} miles USD/semana")
for v in vars_p1:
    print(f"{v.name}: {v.varValue:.4f} GWh")

=== SOLUCIÓN PRIMAL PULP (PROB 1) ===
Estado: Optimal
Costo óptimo: 19870.0000 miles USD/semana
x_G1: 260.0000 GWh
x_G2: 120.0000 GWh
x_G3: 230.0000 GWh
x_G4: 170.0000 GWh
x_G5: 0.0000 GWh


In [4]:
# --- RESOLUCIÓN DEL DUAL DIRECTO COMO UN PL INDEPENDIENTE (SCIPY) ---
# Queremos maximizar la f.obj dual. Para linprog minimizamos el negativo.
# Variables duales: y1 (libre), y2 (<=0), y3 (>=0), y4 (>=0), w1..w5 (<=0)
# x = [y1, y2, y3, y4, w1, w2, w3, w4, w5]

c_dual = -np.array([780, 245000, 312, 50, 260, 120, 300, 280, 90])

# Restricciones duales (todas son <=)
A_ub_dual = np.array([
    [1, 0, 1, 0, 1, 0, 0, 0, 0], # G1
    [1, 0, 1, 0, 0, 1, 0, 0, 0], # G2
    [1, 400, 0, 1, 0, 0, 1, 0, 0], # G3
    [1, 900, 0, 0, 0, 0, 0, 1, 0], # G4
    [1, 700, 0, 0, 0, 0, 0, 0, 1]  # G5
])
b_ub_dual = np.array([12, 8, 45, 32, 95])

bounds_dual = [
    (None, None), # y1
    (None, 0),    # y2
    (0, None),    # y3
    (0, None),    # y4
    (None, 0),    # w1
    (None, 0),    # w2
    (None, 0),    # w3
    (None, 0),    # w4
    (None, 0)     # w5
]

res_p1_dual = linprog(c=c_dual, A_ub=A_ub_dual, b_ub=b_ub_dual, bounds=bounds_dual, method='highs')
print("=== SOLUCIÓN DUAL DIRECTO (PROB 1) ===")
print(f"Objetivo dual óptimo: {-res_p1_dual.fun:.4f} miles USD/semana")
names_dual = ['y_Demanda', 'y_Emisiones', 'y_CuotaRenov', 'y_TakeOrPay', 'w_G1', 'w_G2', 'w_G3', 'w_G4', 'w_G5']
for name, val in zip(names_dual, res_p1_dual.x):
    print(f"{name}: {val:.6f}")

=== SOLUCIÓN DUAL DIRECTO (PROB 1) ===
Objetivo dual óptimo: 19870.0000 miles USD/semana
y_Demanda: 55.400000
y_Emisiones: -0.026000
y_CuotaRenov: 0.000000
y_TakeOrPay: 0.000000
w_G1: -43.400000
w_G2: -47.400000
w_G3: 0.000000
w_G4: 0.000000
w_G5: 0.000000


In [5]:
# --- VERIFICACIÓN DE DUALIDAD FUERTE (PROB 1) ---
primal_val = res_p1_scipy.fun
dual_val = -res_p1_dual.fun
gap_abs = abs(primal_val - dual_val)
gap_rel = gap_abs / (abs(primal_val) + 1e-12)
print(f"Gap Absoluto: {gap_abs:.4e}")
print(f"Gap Relativo: {gap_rel:.4e}")

Gap Absoluto: 0.0000e+00
Gap Relativo: 0.0000e+00



## PROBLEMA 2: Portafolio de Crédito

### 1. Formulación Primal (Matemática)
Sea $x_j$ el monto colocado en la línea de crédito $j$ en millones USD ($j \in \{1, 2, 3, 4, 5\}$ para Consumo, Vehículo, Hipotecario, PyME, Corporativo).
Convertimos el rendimiento nominal a fracción para los cálculos.

**Función Objetivo (Maximizar Rendimiento Neto en MUSD):**
$$\max z = 0.165x_1 + 0.115x_2 + 0.070x_3 + 0.130x_4 + 0.060x_5$$

**Restricciones:**
1. **Presupuesto:**
   $$x_1 + x_2 + x_3 + x_4 + x_5 = 600 \quad (\text{MUSD}) \quad [y_1 \in \mathbb{R}]$$
2. **Apetito de Riesgo:**
   $$9x_1 + 5x_2 + 2x_3 + 6x_4 + 1x_5 \le 3300 \quad (\text{puntos}) \quad [y_2 \ge 0]$$
3. **Regulatorio Bajo Riesgo:**
   $$x_3 + x_5 \ge 120 \iff -x_3 - x_5 \le -120 \quad (\text{MUSD}) \quad [y_3 \le 0]$$
4. **Meta Comercial PyME:**
   $$x_4 \ge 50 \iff -x_4 \le -50 \quad (\text{MUSD}) \quad [y_4 \le 0]$$
5. **Límites de Concentración:**
   $$0 \le x_1 \le 150$$
   $$0 \le x_2 \le 120$$
   $$0 \le x_3 \le 250$$
   $$0 \le x_4 \le 180$$
   $$0 \le x_5 \le 250$$
```

In [6]:
# --- DATOS PROBLEMA 2 ---
c_p2 = -np.array([0.165, 0.115, 0.070, 0.130, 0.060]) # Negativo para maximizar con scipy

# Desigualdades (A_ub * x <= b_ub)
A_ub_p2 = np.array([
    [9, 5, 2, 6, 1],
    [0, 0, -1, 0, -1],
    [0, 0, 0, -1, 0]
])
b_ub_p2 = np.array([3300, -120, -50])

# Igualdad
A_eq_p2 = np.array([[1, 1, 1, 1, 1]])
b_eq_p2 = np.array([600])

bounds_p2 = [
    (0, 150),
    (0, 120),
    (0, 250),
    (0, 180),
    (0, 250)
]

In [7]:
# --- RESOLUCIÓN PRIMAL CON SCIPY (PROB 2) ---
res_p2_scipy = linprog(
    c=c_p2,
    A_ub=A_ub_p2,
    b_ub=b_ub_p2,
    A_eq=A_eq_p2,
    b_eq=b_eq_p2,
    bounds=bounds_p2,
    method='highs'
)

print("=== SOLUCIÓN PRIMAL SCIPY (PROB 2) ===")
print(f"Rendimiento neto óptimo: {-res_p2_scipy.fun:.4f} MUSD")
print(f"Asignación líneas (L1-L5) MUSD: {res_p2_scipy.x}")

=== SOLUCIÓN PRIMAL SCIPY (PROB 2) ===
Rendimiento neto óptimo: 72.1500 MUSD
Asignación líneas (L1-L5) MUSD: [150. 120. 120. 180.  30.]


In [8]:
# --- RESOLUCIÓN PRIMAL CON PULP (PROB 2) ---
prob2 = pulp.LpProblem("Portafolio_Credito", pulp.LpMaximize)

l1 = pulp.LpVariable('x_L1', lowBound=0, upBound=150)
l2 = pulp.LpVariable('x_L2', lowBound=0, upBound=120)
l3 = pulp.LpVariable('x_L3', lowBound=0, upBound=250)
l4 = pulp.LpVariable('x_L4', lowBound=0, upBound=180)
l5 = pulp.LpVariable('x_L5', lowBound=0, upBound=250)

vars_p2 = [l1, l2, l3, l4, l5]

prob2 += 0.165*l1 + 0.115*l2 + 0.070*l3 + 0.130*l4 + 0.060*l5

prob2 += (l1 + l2 + l3 + l4 + l5 == 600, "Presupuesto")
prob2 += (9*l1 + 5*l2 + 2*l3 + 6*l4 + 1*l5 <= 3300, "Apetito_Riesgo")
prob2 += (l3 + l5 >= 120, "Bajo_Riesgo")
prob2 += (l4 >= 50, "Meta_PyME")

prob2.solve(pulp.PULP_CBC_CMD(msg=False))

print("=== SOLUCIÓN PRIMAL PULP (PROB 2) ===")
print(f"Rendimiento neto óptimo: {pulp.value(prob2.objective):.4f} MUSD")
for v in vars_p2:
    print(f"{v.name}: {v.varValue:.4f} MUSD")

=== SOLUCIÓN PRIMAL PULP (PROB 2) ===
Rendimiento neto óptimo: 72.1500 MUSD
x_L1: 150.0000 MUSD
x_L2: 120.0000 MUSD
x_L3: 120.0000 MUSD
x_L4: 180.0000 MUSD
x_L5: 30.0000 MUSD


In [20]:
# --- RESOLUCIÓN DEL DUAL DIRECTO (PROB 2) ---
# Variables duales: y1 (libre), y2 (>=0), y3 (<=0), y4 (<=0), w1..w5 (>=0)
# Buscamos minimizar g_dual = 600y1 + 3300y2 + 120y3 + 50y4 + sum(lim_j * w_j)
c_dual_p2 = np.array([600, 3300, 120, 50, 150, 120, 250, 180, 250])

A_ub_dual_p2 = np.array([
    [1, 9, 0, 0, 1, 0, 0, 0, 0],  # L1 (1y1 + 9y2 + w1 >= c1)
    [1, 5, 0, 0, 0, 1, 0, 0, 0],  # L2 (1y1 + 5y2 + w2 >= c2)
    [1, 2, 1, 0, 0, 0, 1, 0, 0],  # L3 (1y1 + 2y2 + 1y3 + w3 >= c3) <-- Corrected y3 coeff
    [1, 6, 0, 1, 0, 0, 0, 1, 0],  # L4 (1y1 + 6y2 + 1y4 + w4 >= c4) <-- Corrected y4 coeff
    [1, 1, 1, 0, 0, 0, 0, 0, 1]   # L5 (1y1 + 1y2 + 1y3 + w5 >= c5) <-- Corrected y3 coeff
])
b_ub_dual_p2 = np.array([0.165, 0.115, 0.070, 0.130, 0.060])

bounds_dual_p2 = [
    (None, None), # y1
    (0, None),    # y2
    (None, 0),    # y3
    (None, 0),    # y4
    (0, None),    # w1
    (0, None),    # w2
    (0, None),    # w3
    (0, None),    # w4
    (0, None)     # w5
]

res_p2_dual = linprog(c=c_dual_p2, A_ub=-A_ub_dual_p2, b_ub=-b_ub_dual_p2, bounds=bounds_dual_p2, method='highs')
print("=== SOLUCIÓN DUAL DIRECTO (PROB 2) ===")

if res_p2_dual.success:
    print(f"Objetivo dual óptimo: {res_p2_dual.fun:.6f} MUSD")
    names_dual_p2 = ['y_Presupuesto', 'y_ApetitoRiesgo', 'y_BajoRiesgo', 'y_MetaPyME', 'w_L1', 'w_L2', 'w_L3', 'w_L4', 'w_L5']
    for name, val in zip(names_dual_p2, res_p2_dual.x):
        print(f"{name}: {val:.6f}")
else:
    print(f"Linprog no encontró una solución óptima.")
    print(f"Status: {res_p2_dual.status}")
    print(f"Message: {res_p2_dual.message}")

=== SOLUCIÓN DUAL DIRECTO (PROB 2) ===
Objetivo dual óptimo: 72.150000 MUSD
y_Presupuesto: 0.050000
y_ApetitoRiesgo: 0.010000
y_BajoRiesgo: 0.000000
y_MetaPyME: 0.000000
w_L1: 0.025000
w_L2: 0.015000
w_L3: 0.000000
w_L4: 0.020000
w_L5: 0.000000



## SECCIÓN V: Verificación y Análisis Transversal

### V.1 Verificación de Factibilidad Primal sin Solver (Residuos)


In [16]:
# --- VERIFICACIÓN DE RESIDUOS PRIMALES CON NUMPY ---
tol = 1e-8

# Problema 1
x1_opt = res_p1_scipy.x
residuos_eq_1 = A_eq_p1 @ x1_opt - b_eq_p1
residuos_ub_1 = A_ub_p1 @ x1_opt - b_ub_p1
print("--- Residuos Primales Problema 1 ---")
print(f"Demanda (Eq) residuo: {residuos_eq_1[0]:.2e} (Tolera: {abs(residuos_eq_1[0]) < tol})")
print(f"Emisiones (Ub) holgura: {-residuos_ub_1[0]:.4f} tCO2")
print(f"Cuota Renov. (Ub) exceso: {-residuos_ub_1[1]:.4f} GWh")
print(f"Take-or-pay (Ub) exceso: {-residuos_ub_1[2]:.4f} GWh")

# Problema 2
x2_opt = res_p2_scipy.x
residuos_eq_2 = A_eq_p2 @ x2_opt - b_eq_p2
residuos_ub_2 = A_ub_p2 @ x2_opt - b_ub_p2
print("\n--- Residuos Primales Problema 2 ---")
print(f"Presupuesto (Eq) residuo: {residuos_eq_2[0]:.2e} (Tolera: {abs(residuos_eq_2[0]) < tol})")
print(f"Apetito Riesgo (Ub) holgura: {-residuos_ub_2[0]:.4f} puntos")
print(f"Bajo Riesgo (Ub) exceso: {-residuos_ub_2[1]:.4f} MUSD")
print(f"Meta PyME (Ub) exceso: {-residuos_ub_2[2]:.4f} MUSD")

--- Residuos Primales Problema 1 ---
Demanda (Eq) residuo: 0.00e+00 (Tolera: True)
Emisiones (Ub) holgura: -0.0000 tCO2
Cuota Renov. (Ub) exceso: 68.0000 GWh
Take-or-pay (Ub) exceso: 180.0000 GWh

--- Residuos Primales Problema 2 ---
Presupuesto (Eq) residuo: 0.00e+00 (Tolera: True)
Apetito Riesgo (Ub) holgura: -0.0000 puntos
Bajo Riesgo (Ub) exceso: 30.0000 MUSD
Meta PyME (Ub) exceso: 130.0000 MUSD



### V.5 Análisis de Sensibilidad por Perturbación Finita (Cálculo de Rangos)


In [17]:
# --- ANÁLISIS DE SENSIBILIDAD POR PERTURBACIÓN (PROB 1: DEMANDA) ---
deltas = np.linspace(-50, 50, 21)
resultados_demanda = []

for d in deltas:
    b_eq_temp = np.array([780 + d])
    res_temp = linprog(
        c=c_p1, A_ub=A_ub_p1, b_ub=b_ub_p1, A_eq=A_eq_p1, b_eq=b_eq_temp,
        bounds=bounds_p1, method='highs'
    )
    if res_temp.success:
        resultados_demanda.append((780 + d, res_temp.fun, res_temp.x))

df_sens_demanda = pd.DataFrame(resultados_demanda, columns=['Demanda_GWh', 'Costo_Optimo', 'Asignacion'])
display(df_sens_demanda.head())

,Demanda_GWh,Costo_Optimo,Asignacion
0,730.0,17100.0,"[260.0, 120.0, 139.99999999999997, 210.0, 0.0]"
1,735.0,17377.0,"[260.0, 120.0, 148.99999999999997, 206.0, 0.0]"
2,740.0,17654.0,"[260.0, 120.0, 157.99999999999997, 202.0, 0.0]"
3,745.0,17931.0,"[260.0, 120.0, 166.99999999999997, 198.0, 0.0]"
4,750.0,18208.0,"[260.0, 120.0, 175.99999999999997, 194.0, 0.0]"


In [18]:
# --- ANÁLISIS DE SENSIBILIDAD POR PERTURBACIÓN (PROB 2: APETITO DE RIESGO) ---
deltas_p2 = np.linspace(-300, 300, 21)
resultados_riesgo = []

for d in deltas_p2:
    b_ub_temp = b_ub_p2.copy()
    b_ub_temp[0] = 3300 + d
    res_temp = linprog(
        c=c_p2, A_ub=A_ub_p2, b_ub=b_ub_temp, A_eq=A_eq_p2, b_eq=b_eq_p2,
        bounds=bounds_p2, method='highs'
    )
    if res_temp.success:
        # Convertimos costo a positivo ya que es maximización
        resultados_riesgo.append((3300 + d, -res_temp.fun, res_temp.x))

df_sens_riesgo = pd.DataFrame(resultados_riesgo, columns=['Límite_Riesgo', 'Rendimiento_Optimo', 'Asignacion'])
display(df_sens_riesgo.head())

,Límite_Riesgo,Rendimiento_Optimo,Asignacion
0,3000.0,68.58750,"[127.5, 120.0, 0.0, 180.0, 172.5]"
1,3030.0,68.98125,"[131.25, 120.0, 0.0, 180.0, 168.75]"
2,3060.0,69.37500,"[135.0, 120.0, 0.0, 180.0, 165.0]"
3,3090.0,69.76875,"[138.75, 120.0, 0.0, 180.0, 161.25]"
4,3120.0,70.16250,"[142.5, 120.0, 0.0, 180.0, 157.5]"



### V.8 Documentación de Limitaciones del Código y Supuestos (Uso de IA)

* **Limitación en la Extrapolación Dual en PuLP:** Al resolver modelos con cotas de variables directas en el constructor de variables (`lowBound` / `upBound`), PuLP no reporta explícitamente los valores de los costos reducidos ni los multiplicadores de los límites individuales en las restricciones duales de la misma forma estructurada que SciPy. Para corregir esto, se requirió implementar la formulación dual explícita como un PL separado.
* **Supuesto de Linealidad:** Asumimos que los costos unitarios de generación eléctrica y los rendimientos de crédito permanecen constantes independientemente del volumen total asignado (sin economías de escala ni tasas escalonadas).
